The same lab as a web page, with the folded answers: https://ml.wxing.me/materials/lesson7/lab.html

# Lab 7: Use PCA, and Catch It When It Goes Wrong
## MPS311/439 Machine Learning
**Lesson 7 Lab Session | Core: about 43 minutes**

## What this lab is for

The lecture explained why PCA works. Today you use it as a tool, on three jobs: look at data with 30 measurements, make QDA work on 784 pixels, and check code that an AI wrote. All the code is given. You run it, change the value marked `← change`, and decide what the numbers tell you. After Parts 1 to 3, **write down** one number from your output and one sentence.

Each Part ends with a folded **Going further** box: open questions with no single answer, for when you finish early. The folded **Try with AI** boxes are also for when you finish early; the **Advanced Study** is for after class.

## How to work

1. This notebook already has all the code. Save a copy first (**File → Save a copy in Drive**). Run the cells from top to bottom: later cells use names made by earlier ones.
2. Write your answers in a text cell (**+ Text**) under the code.
3. **Run, and you should see** tells you what your output should show, so you know you are on track.

## Part 0: Set up (3 minutes)

Run this cell without reading it closely. It is the same at the start of every lab, with today's two new tools at the end.

In [ ]:
import numpy as np                                        # numbers and arrays
import pandas as pd                                       # tables
import matplotlib.pyplot as plt                           # plots
import time                                               # a clock, to time the fitting
import urllib.request                                     # to download a file
from sklearn.datasets import load_breast_cancer           # the first dataset
from sklearn.model_selection import train_test_split, KFold, cross_val_score   # splitting and cross-validation
from sklearn.preprocessing import StandardScaler          # puts every column on one scale
from sklearn.linear_model import LogisticRegression       # a straight-line classifier
from sklearn.pipeline import make_pipeline                # chains two steps into one model (Going further)
from sklearn.decomposition import PCA                     # today's tool
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis   # QDA

## Part 1: Look at data with 30 measurements (10 minutes)

**The job.** 569 breast tumours (the Wisconsin breast cancer data that comes with scikit-learn, a teaching dataset), each with 30 measurements from a microscope image (radius, area, smoothness, concavity, …). You cannot draw 30 axes. Before building any classifier, you want one picture: do malignant and benign tumours look different?

In [ ]:
data = load_breast_cancer()                               # ← change: which dataset
X = data.data                                             # the 30 measurements (features)
y = 1 - data.target                                       # ← change: the labels (here 1 = malignant, 0 = benign)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

**Run, and you should see:** `(426, 30) [267 159]`.

Keep two components and draw every training tumour as a point. PCA does not see `y_train`; the colours are added only for the picture.

In [ ]:
pca = PCA(n_components=2, random_state=0)                 # keep two components
Z = pca.fit_transform(X_train)                            # ← change: X_train or X_train_s
print(pca.explained_variance_ratio_.round(3))             # the share of the spread each component keeps
weights = pd.Series(pca.components_[0], index=data.feature_names)   # the 30 weights of the first component, with names
sizes = weights.abs()                                     # the size of each weight, ignoring its sign
largest = sizes.sort_values(ascending=False)              # largest first
print(largest.head(3).round(2))                           # the three largest weights
plt.scatter(Z[:, 0], Z[:, 1], c=y_train, cmap="coolwarm", s=10)     # one dot per tumour: red malignant, blue benign
plt.xlabel("first component")                             # label the axes
plt.ylabel("second component")                            # the vertical axis
plt.show()                                                # draw the picture

Run it as it is, then change `X_train` to `X_train_s` (the standardised columns) and run again.

**Write down:** for each run, the share of the first component and the column with the largest weight. In one sentence: which picture would you show a doctor, and why? (Does the picture use all 30 measurements, or is one column doing all the work?)

*Your answer:* 

**Think about** (with your neighbour, no writing): someone shows you a PCA picture where the first component keeps 98%. What would you check before you believe it?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the 30 weights of the first component (standardised data) as a horizontal bar chart, with the column names.
> 2. **Tell the AI:** `weights` is a pandas Series of 30 numbers indexed by the column names.
> 3. **Ask:** "Write simple Python that draws `weights`, sorted by value, as a horizontal bar chart with matplotlib."
> 4. **Check:** the longest bars are the concave points and concavity columns, about 0.26; almost every bar points the same way.

> **Going further (optional)**
>
> - How many components do you need to keep 90% of the spread of the standardised data? Change `n_components=2` to `n_components=None` (keep them all) and print `pca.explained_variance_ratio_.cumsum().round(3)`.
> - How much does a classifier lose if it sees only the two numbers of your picture? `make_pipeline` chains PCA and logistic regression into one model, so cross-validation fits PCA on the training folds only:
> 
>

In [ ]:
two = make_pipeline(PCA(n_components=2, random_state=0), LogisticRegression(max_iter=5000))  # ← change: the number of components; PCA is fitted again inside each fold
print(cross_val_score(two, X_train_s, y_train, cv=kf).mean())   # 5-fold cross-validated accuracy
full = LogisticRegression(max_iter=5000)                  # all 30 standardised columns
print(cross_val_score(full, X_train_s, y_train, cv=kf).mean())  # the same 5 folds

>   When would the two-number model be worth the loss?
> - Read the 30 weights of the first component (the Try with AI chart helps). What single property of a tumour does it measure?


## Part 2: Make QDA work on 784 pixels (12 minutes)

**The job.** Assignment 2 asks for QDA on handwritten digits. First load them with the same data block, then split off a validation set for choosing (the same three parts as the assignment).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/wayXing/mps311-439-course-materials/main/lecture-slids/lessons/data/mnist10k.npz"
urllib.request.urlretrieve(DATA_URL, "mnist10k.npz")      # 10,000 handwritten digits, about 1.6 MB (if this fails, see the Appendix)
data = np.load("mnist10k.npz")                            # ← change: which dataset
X = data["X"] / 255                                       # the 784 pixels of each 28 x 28 image, scaled to 0 to 1
y = data["y"]                                             # ← change: the labels (the digit, 0 to 9)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column (Part 3 asks if we need it)
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=0, stratify=y_train)   # fit on 3/4, choose on 1/4
print(X_fit.shape, X_val.shape, X_test.shape)             # fitting, validation, test

**Run, and you should see:** `(7500, 784)` with between 680 and 851 of each digit, then `(5625, 784) (1875, 784) (2500, 784)`.

Now try QDA on the pixels directly:

In [ ]:
qda = QuadraticDiscriminantAnalysis()                     # one covariance matrix per digit
qda.fit(X_fit, y_fit)                                     # on all 784 pixels
print(qda.score(X_val, y_val))                            # validation accuracy

**Run, and you should see:** a message that the covariance matrix of a class "is not full rank". Depending on your version of scikit-learn, it is an error that stops the cell, or a warning followed by a validation accuracy of about `0.73`, far too low for digits. Both say the same thing. A red error here is expected: read it, then go on to the next cell. QDA estimates a 784 × 784 covariance matrix for each digit from only about 560 images: fewer images than pixels, and many pixels (the corners) are 0 in every image. The matrix cannot be inverted, so QDA cannot be fitted properly.

PCA gives QDA a few numbers per image instead of 784. Learn the components from the fitting images only, give QDA the K scores, and score it on the validation images:

In [ ]:
for k in [5, 10, 20, 40, 50]:                             # ← change: the numbers of components to try
    pca = PCA(n_components=k, random_state=0)             # keep k components
    pca.fit(X_fit)                                        # learn the mean image and the components from the fitting images only
    Z_fit = pca.transform(X_fit)                          # k scores for each fitting image
    Z_val = pca.transform(X_val)                          # the same k directions for the validation images
    qda = QuadraticDiscriminantAnalysis()                 # a new QDA model for this k
    qda.fit(Z_fit, y_fit)                                 # QDA on the k scores
    print(k, round(pca.explained_variance_ratio_.sum(), 3), round(qda.score(Z_val, y_val), 3))   # k, share kept, validation accuracy

**Write down:** the K you choose (if two are within 0.002, take the smaller), its validation accuracy and the share of the spread it keeps. Is your K at an end of the list? If it is, add a larger K to the list and run again.

*Your answer:* 

**Think about** (with your neighbour, no writing): PCA never looked at the labels. How can it still help a classifier?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the validation accuracy of QDA and of logistic regression against K, two lines on one plot.
> 2. **Tell the AI:** `X_fit`, `y_fit`, `X_val`, `y_val`, the K values `[5, 10, 20, 40, 50]`, and that PCA must be fitted on `X_fit` only.
> 3. **Ask:** "Write simple Python, one step per line, that loops over the K values, fits PCA on `X_fit`, fits QDA and LogisticRegression(max_iter=2000) on the scores, and plots the two validation accuracies against K."
> 4. **Check:** the QDA line reaches 0.957 at K = 40; the logistic regression line is 0.897 at K = 40 and stays below the QDA line from K = 10 on.

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the first 6 principal components as 28 × 28 images.
> 2. **Tell the AI:** `pca` is a fitted PCA; `pca.components_` has one row of 784 numbers per component.
> 3. **Ask:** "Write simple Python that reshapes each of the first 6 rows of `pca.components_` to 28 × 28 and shows them side by side with `cmap='RdBu_r'`, with 0 in the middle of the colour scale."
> 4. **Check:** six 28 × 28 images; the first is a round shape, a ring (its colours may be swapped: the sign of a component can flip).

> **Going further (optional)**
>
> - Add `60`, `80` and `100` to the list of K. What happens from 80 on, and why does a larger K bring back the problem PCA solved? Is K = 60 any better than 40?
> - PCA also makes models faster. Time logistic regression on the 784 pixels and on the 40 scores (the first fit takes a while and may print a convergence warning; that is fine):
> 
>

In [ ]:
pca = PCA(n_components=40, random_state=0)                # ← change: the K you chose
pca.fit(X_fit)                                            # fitting images only
Z_fit = pca.transform(X_fit)                            # 40 scores for each fitting image
Z_val = pca.transform(X_val)                            # the same directions for the validation images
start = time.time()                                       # the clock before fitting
line = LogisticRegression(max_iter=2000)                # a straight-line classifier
line.fit(X_fit, y_fit)                                    # on the 784 pixels
print("784 pixels:", round(time.time() - start, 2), "s", round(line.score(X_val, y_val), 3))  # seconds, validation accuracy
start = time.time()                                     # the clock again
line.fit(Z_fit, y_fit)                                    # on the K scores
print("K scores:", round(time.time() - start, 2), "s", round(line.score(Z_val, y_val), 3))  # seconds, validation accuracy

>   How many times faster are the scores, and how much does the accuracy change?
> - Change `test_size=0.25` in the validation split to `0.2` and `0.3`, and run the loop with K = 40, 50 and 60. Does the best K stay the same? How large are the differences between these K compared with the change from one split to another?


## Part 3: Check the code an AI wrote (16 minutes)

**The job.** You asked an AI for "PCA with 40 components, then QDA, scored on the validation set". It gave three versions. Each changes something in the code of Part 2, and each runs without an error. First run the Part 2 code for K = 40 once more, so you have it next to the three versions:

In [ ]:
pca = PCA(n_components=40, random_state=0)                # keep 40 components
pca.fit(X_fit)                                            # learn the components from the fitting images only
Z_fit = pca.transform(X_fit)                              # scores for the fitting images
Z_val = pca.transform(X_val)                              # the same directions for the validation images
qda = QuadraticDiscriminantAnalysis()                     # a new QDA model
qda.fit(Z_fit, y_fit)                                     # QDA on the scores
print(round(qda.score(Z_val, y_val), 3))                  # validation accuracy

**Run, and you should see:** `0.957`.

**Version A**

In [ ]:
pca = PCA(n_components=40, random_state=0)                # keep 40 components
pca.fit(X_fit)                                            # learn the components
Z_fit = pca.transform(X_fit)                              # scores for the fitting images
Z_val = pca.fit_transform(X_val)                          # scores for the validation images
qda = QuadraticDiscriminantAnalysis()                     # a new QDA model
qda.fit(Z_fit, y_fit)                                     # QDA on the scores
print(round(qda.score(Z_val, y_val), 3))                  # validation accuracy

**Version B**

In [ ]:
X_all = np.concatenate([X_fit, X_val, X_test])            # all 10,000 images
pca = PCA(n_components=40, random_state=0)                # keep 40 components
pca.fit(X_all)                                            # learn the components
Z_fit = pca.transform(X_fit)                              # scores for the fitting images
Z_val = pca.transform(X_val)                              # scores for the validation images
qda = QuadraticDiscriminantAnalysis()                     # a new QDA model
qda.fit(Z_fit, y_fit)                                     # QDA on the scores
print(round(qda.score(Z_val, y_val), 3))                  # validation accuracy

**Version C**

In [ ]:
scaler_pixels = StandardScaler()                          # one scale for every pixel
X_fit_s = scaler_pixels.fit_transform(X_fit)              # learn the scale from the fitting images
X_val_s = scaler_pixels.transform(X_val)                  # the same scale for the validation images
pca = PCA(n_components=40, random_state=0)                # keep 40 components
pca.fit(X_fit_s)                                          # learn the components
Z_fit = pca.transform(X_fit_s)                            # scores for the fitting images
Z_val = pca.transform(X_val_s)                            # scores for the validation images
qda = QuadraticDiscriminantAnalysis()                     # a new QDA model
qda.fit(Z_fit, y_fit)                                     # QDA on the scores
print(round(qda.score(Z_val, y_val), 3))                  # validation accuracy

**Write down:** one line per version: its validation accuracy, then *wrong* or *allowed but worse*, and a few words why. One of the three scores is very low: that is the real result, not a copying mistake. Do not judge by the score alone; for each version ask: **which images did PCA learn from?** Only the fitting images is allowed. If so, and the score is lower than 0.957, the version is *allowed but worse*.

*Your answer:* 

**Think about** (with your neighbour, no writing): Version B scored as well as your own code. If you had not read it line by line, how could you have noticed?

> **Going further (optional)**
>
> - Write your own version D that differs from Part 2 in one line, runs without an error and is wrong (for example, something to do with `X_test` or `y_val`). Give it to your neighbour. Can they find the line from the code alone? From the score alone?
> - In Version C, look at `scaler_pixels.scale_` (the spread of each pixel, learned from the fitting images). How many pixels have a spread below 0.01? What happens to them after dividing by it?


## Part 4: The test set, once (2 minutes)

Use the K you chose in Part 2 and read the test set once.

In [ ]:
pca = PCA(n_components=40, random_state=0)                # ← change: the K you chose in Part 2
pca.fit(X_fit)                                            # fitting images only
Z_fit = pca.transform(X_fit)                              # scores for the fitting images
Z_test = pca.transform(X_test)                            # the test images: transform only
qda = QuadraticDiscriminantAnalysis()                     # a new QDA model
qda.fit(Z_fit, y_fit)                                     # QDA on the scores
print(qda.score(Z_test, y_test))                          # the 2,500 test images, read once

**Run, and you should see:** about `0.962` at K = 40. Do not change K after seeing this number: that would be choosing on the test set.

# After class

## How today's lab helps with Assignment 2

| What you did today | Where it appears in Assignment 2 |
|---|---|
| Part 2: QDA fails on the pixels; PCA fitted on the fitting images only | Section 3: "**Fit PCA on the training set only**, then use it to transform the training and validation sets" |
| Part 2: the loop over K | Section 3: use the same loop with the assignment's `X_train`, `X_val`, K = 10, 50 and 100, and add logistic regression, LDA and the decision tree; `pca.explained_variance_ratio_.sum()` is the column of cumulative variance in Table 3. Use the `reg_param` you chose in Section 2 for QDA. Today QDA broke down at large K because each digit had only about 560 images; the assignment has about 5,000 per digit |
| Part 3: the three versions | Section 3 and 4: none of these mistakes may appear in your code; `test` is only transformed, once |
| Try with AI: the components as images | Section 3, Going further: "Turn the first few principal components back into 28 × 28 images" |

## Reflection

One or two sentences each.

1. What does PCA's `.fit()` learn, and where does it keep it?
2. Version B of Part 3 scored as well as the correct code. Why is it still wrong?
3. When should you standardise before PCA, and when not? Use Part 1 and Part 3 as your examples.

# Appendix: only if the digits did not download

Use this only if the first cell of Part 2 stopped with a download error. Each cell below gives exactly the same 10,000 images; run one, then go on with the rest of the data block (from `X = data["X"] / 255`).

**Backup 1: a second copy of the course file**, on another server.

In [ ]:
BACKUP_URL = "https://cdn.jsdelivr.net/gh/wayXing/mps311-439-course-materials@main/lecture-slids/lessons/data/mnist10k.npz"
urllib.request.urlretrieve(BACKUP_URL, "mnist10k.npz")    # the same file, from a different server
data = np.load("mnist10k.npz")                            # the file holds two arrays, X and y
print(data["X"].shape)                                    # images and pixels

**Backup 2: the official MNIST file** (the one Keras uses), from which the course picked its 10,000 images.

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
urllib.request.urlretrieve(MNIST_URL, "mnist.npz")        # all 70,000 MNIST images, about 11 MB
full = np.load("mnist.npz")                               # x_train holds the 60,000 training images
rng = np.random.default_rng(0)                            # the random numbers the course used
pick = rng.choice(60000, 10000, replace=False)            # 10,000 of the 60,000 training images
pick = np.sort(pick)                                      # in their original order
images = full["x_train"][pick]                            # the chosen images, 28 x 28 each
X_course = images.reshape(10000, 784)                     # one row of 784 pixels per image
y_course = full["y_train"][pick]                          # their labels
data = {"X": X_course, "y": y_course}                     # the same two arrays as the course file
print(data["X"].shape)                                    # images and pixels

**Run, and you should see:** `(10000, 784)` from either cell.